# GROUP PROJECT

**IMPORTANT from classes**

* try remove shit data: create 3 folds and record on what photos model fails - they are the condidats to be removed
* try ebmedinng model and find outliers in that space for each cluster
* mb try base models from hugging face 

* use tensorboard
* add a heatmap on top of image at the end
* plot the closest images

## Importing libraries and setting up google drive

In [ ]:
# ============================================================================
# IMPORTS - Deep Learning Rare Animals Classification
# ============================================================================

# Core Python
import os
import ast
import math
import datetime
from collections import Counter

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Image, display

# Image processing
from PIL import Image as PilImage
import cv2
from tqdm import tqdm

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow/Keras - Core
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from tensorflow.keras.utils import to_categorical

# TensorFlow/Keras - Layers
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Flatten,
    Conv2D, MaxPooling2D, GlobalAveragePooling2D, 
    BatchNormalization, Concatenate, Add, Activation, Embedding,
    LeakyReLU
)

# TensorFlow/Keras - Optimizers & Regularizers
from tensorflow.keras.optimizers import Adam, SGD, Adagrad, Adadelta, RMSprop
from tensorflow.keras.regularizers import L1, L2

# TensorFlow/Keras - Applications (Pretrained models)
from tensorflow.keras.applications import (
    EfficientNetB0, EfficientNetB4, InceptionV3
)
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

# Mixed Precision & Optimization
from tensorflow.keras.mixed_precision import set_global_policy

# YOLO for object detection
from ultralytics import YOLO

# CLIP for semantic analysis
from transformers import CLIPProcessor, CLIPModel
import torch

# Focal Loss for class imbalance
from focal_loss import SparseCategoricalFocalLoss

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress INFO and WARNING messages

In [ ]:
directory = '/workspace/rare_species/'

In [ ]:
# Place this at the very top of your script/notebook
tf.config.optimizer.set_jit(True)
# Alternatively, compile your model with the jit_compile flag
# model.compile(jit_compile=True, ...)

In [ ]:
# Enable Mixed Precision policy at the start
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')
# Make sure the final output layer is still float32 for stability
# final_layer = Dense(num_classes, activation='softmax', dtype='float32')(previous_layer)

In [ ]:
# # Set up google drive FOR COLAB
# from google.colab.patches import cv2_imshow
# from google.colab import drive
# drive.mount('/content/drive/')

# # The directory where all the files will go
# !unzip -o "/content/drive/MyDrive/DeepLearningProject/rare_species.zip" -d {directory} > /dev/null # remove all the prints (takes +-2min yo run)

In [ ]:
# import torch
# print("CUDA available:", torch.cuda.is_available())
# print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices('GPU'))

In [ ]:
# tf.debugging.set_log_device_placement(True)     # to see which device is being used

In [ ]:
metadata = pd.read_csv(f'{directory}metadata.csv')

## Custom functions (later to .py)

In [ ]:
def explore_image_files(file_paths, explore_values=False):
    # Define the variables of smalles and biggest images
    image_sizes = []
    color_channels = []
    formats = []
    min_vals = []
    max_vals = []
    ratios = []

    # Iteration loop for each folder to compare the image sizes

    for file_path in file_paths:
        with PilImage.open(file_path) as img:
            image_sizes.append(img.size)
            color_channels.append(img.mode)
            formats.append(img.format)
            ratios.append(img.size[0]/img.size[1])

            if explore_values:
                # Convert to numpy to check the actual data type, Takes alot of time
                img_array = np.array(img)

                # Value range
                min_vals.append(img_array.min())
                max_vals.append(img_array.max())

    if explore_values:
        return image_sizes, color_channels, formats, ratios, min_vals, max_vals
    else:
        return image_sizes, color_channels, formats, ratios

## Initial exploration

### Metadata exploration

#### Basic exploration

In [ ]:
metadata.info()

In [ ]:
metadata.head()

In [ ]:
metadata.describe(include='O')

In [ ]:
# Get phylum counts
phylum_counts = metadata['phylum'].value_counts()

# Calculate number of families per phylum
families_per_phylum = metadata.groupby('phylum')['family'].nunique()

# Create custom hover data with family counts
hover_data = []
for phylum in phylum_counts.index:
    image_count = phylum_counts[phylum]
    family_count = families_per_phylum[phylum]
    percentage = (image_count / len(metadata)) * 100
    hover_data.append([image_count, percentage, family_count])

# Convert to numpy array for easy indexing
hover_data = np.array(hover_data)

# Create figure
fig = go.Figure(go.Bar(
    x=phylum_counts.index,
    y=phylum_counts.values,
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=phylum_counts.values,
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='<b>%{x}</b><br>' +
                  'Images: %{y}<br>' +
                  'Families: %{customdata[2]}<br>' +
                  'Percentage: %{customdata[1]:.2f}%<extra></extra>',
    customdata=hover_data
))

fig.update_layout(
    title={
        'text': '<b>Distribution of Species by Phylum</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Phylum',
    yaxis_title='Count',
    height=600,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickfont=dict(size=11),
        showgrid=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    showlegend=False
)

fig.show()

# Print summary
print("\n" + "="*60)
print("PHYLUM DISTRIBUTION WITH FAMILY COUNTS")
print("="*60)
for phylum in phylum_counts.index:
    image_count = phylum_counts[phylum]
    family_count = families_per_phylum[phylum]
    pct = (image_count / len(metadata)) * 100
    avg_images_per_family = image_count / family_count
    print(f"{phylum:20s}: {image_count:4d} images ({pct:5.2f}%) | {family_count:3d} families | Avg: {avg_images_per_family:.1f} images/family")
print("="*60)

In [ ]:
# Get all family counts
family_counts = metadata['family'].value_counts()

# Create figure with scrollable y-axis
fig = go.Figure(go.Bar(
    x=family_counts.values,
    y=family_counts.index,
    orientation='h',
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=family_counts.values,
    textposition='outside',
    textfont=dict(size=10),
    hovertemplate='<b>%{y}</b><br>Images: %{x}<br>Percentage: %{customdata:.2f}%<extra></extra>',
    customdata=(family_counts.values / len(metadata)) * 100
))

fig.update_layout(
    title={
        'text': '<b>Distribution Of Species By Family</b><br><sub>All families shown - scroll to explore</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Number of Images',
    yaxis_title='Family',
    height=max(1000, len(family_counts) * 15),  # Dynamic height based on number of families
    width=1200,
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis=dict(
        autorange='reversed',  # Highest count on top
        tickfont=dict(size=9),
        showgrid=False
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    margin=dict(l=200, r=100, t=100, b=50),  # More space for family names
    showlegend=False
)

fig.show()


In [ ]:

# Count at each taxonomic level
phylum_to_family = metadata.groupby(['phylum', 'family']).size().reset_index(name='count')

# Prepare data for Sankey
labels = list(metadata['phylum'].unique()) + list(metadata['family'].unique())
label_dict = {label: idx for idx, label in enumerate(labels)}

source = []
target = []
value = []

for _, row in phylum_to_family.iterrows():
    source.append(label_dict[row['phylum']])
    target.append(label_dict[row['family']])
    value.append(row['count'])

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color='#6366f1'
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color='rgba(99, 102, 241, 0.3)'
    )
)])

fig.update_layout(
    title='<b>Taxonomic Hierarchy: Phylum → Family</b>',
    font=dict(size=12),
    height=800,
    width=1200
)

fig.show()

print("\n📊 TAXONOMIC DIVERSITY:")
print(f"Phylums: {metadata['phylum'].nunique()}")
print(f"Families: {metadata['family'].nunique()}")
print(f"Average families per phylum: {metadata['family'].nunique() / metadata['phylum'].nunique():.1f}")

#### Retracting more information from the images

In [ ]:
# # Extract more information about the images and save to the metadata df (6min)
# metadata['image_size'], metadata['color_channel'], metadata['format'], metadata['aspect_ratio'], \
# metadata['min_val'], metadata['max_val'] = explore_image_files(directory + metadata['file_path'], explore_values=True)

# Don't explore values to save time (1min)
metadata['image_size'], metadata['color_channel'], metadata['format'], metadata['aspect_ratio'] = explore_image_files(directory + metadata['file_path'])

In [ ]:
metadata['width'], metadata['height'] = zip(*metadata['image_size'])

In [ ]:
metadata.describe()

huge file at "\rare_species\mollusca_cardiidae\30003931_46473744_eol-full-size-copy.jpg"

In [ ]:
# Show largest and smallest files
image_files = []
for root, dirs, files in os.walk(directory):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            filepath = os.path.join(root, file)
            try:
                img = PilImage.open(filepath)
                pixel_count = img.size[0] * img.size[1]  # width * height
                image_files.append((filepath, pixel_count, img.size))
                img.close()
            except:
                pass

# Sort by pixel count
if image_files:
    image_files.sort(key=lambda x: x[1], reverse=True)
    largest_image_path, largest_pixels, largest_dims = image_files[0]
    smallest_image_path, smallest_pixels, smallest_dims = image_files[-1]
    
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Display largest image
    img_largest = PilImage.open(largest_image_path)
    axes[0].imshow(img_largest)
    axes[0].axis('off')
    axes[0].set_title(f"LARGEST IMAGE\n{os.path.basename(largest_image_path)}\n{largest_dims[0]} x {largest_dims[1]} pixels\n({largest_pixels:,} total pixels)", fontsize=12, fontweight='bold')
    
    # Display smallest image
    img_smallest = PilImage.open(smallest_image_path)
    axes[1].imshow(img_smallest)
    axes[1].axis('off')
    axes[1].set_title(f"SMALLEST IMAGE\n{os.path.basename(smallest_image_path)}\n{smallest_dims[0]} x {smallest_dims[1]} pixels\n({smallest_pixels:,} total pixels)", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Largest:  {largest_image_path}")
    print(f"          {largest_dims[0]} x {largest_dims[1]} = {largest_pixels:,} pixels")
    print(f"\nSmallest: {smallest_image_path}")
    print(f"          {smallest_dims[0]} x {smallest_dims[1]} = {smallest_pixels:,} pixels")
else:
    print("Error")

- We see a normal looking smallest image, but the largest image looks like it could be split into 4 

## Colour exploration

In [ ]:
# Plot colour channel distribution
color_channel_mapping = {
    'L': 'Greyscale',
    'RGB': 'RGB',
    'RGBA': 'RGBA',
    'P': 'Palette',
    'CMYK': 'CMYK',
    '1': 'Binary',
    'LA': 'Greyscale + Alpha'
}

# Get value counts and map to readable names
color_counts = metadata['color_channel'].value_counts()
color_counts.index = color_counts.index.map(lambda x: color_channel_mapping.get(x, x))

# Create figure
fig = go.Figure(go.Bar(
    x=color_counts.index,
    y=color_counts.values,
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=color_counts.values,
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='<b>%{x}</b><br>Images: %{y}<br>Percentage: %{customdata:.2f}%<extra></extra>',
    customdata=(color_counts.values / len(metadata)) * 100
))

fig.update_layout(
    title={
        'text': '<b>Distribution Of Colour Channels</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Color Channel',
    yaxis_title='Count',
    height=600,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickfont=dict(size=11),
        showgrid=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    showlegend=False
)

fig.show()

In [ ]:
# Custom lines based on previous discoveries
common_ratio = 4/3
width_max = 500        # MB CHANGE LATER

In [ ]:
# Create figure with custom styling
fig = go.Figure()

# Add scatter plot with better styling
fig.add_trace(go.Scatter(
    x=metadata['width'],
    y=metadata['height'],
    mode='markers',
    marker=dict(
        size=4,
        color='#6366f1',
        opacity=0.4,
        line=dict(width=0)
    ),
    name='Images',
    hovertemplate='<b>Width:</b> %{x}px<br><b>Height:</b> %{y}px<extra></extra>'
))

# Add aspect ratio reference line (4:3)
max_y = metadata['height'].max()
max_x_for_ratio = common_ratio * max_y
fig.add_trace(go.Scatter(
    x=[0, max_x_for_ratio],
    y=[0, max_y],
    mode='lines',
    line=dict(color='#ef4444', width=2.5, dash='dash'),
    name='4:3 Aspect Ratio',
    hoverinfo='skip'
))

# Add width threshold line
fig.add_trace(go.Scatter(
    x=[width_max, width_max],
    y=[0, max_y],
    mode='lines',
    line=dict(color='#10b981', width=2.5, dash='dash'),
    name=f'Max Width ({width_max}px)',
    hoverinfo='skip'
))

# Update layout with legend on the right and bold title
fig.update_layout(
    title={
        'text': '<b>Image Size Distribution Analysis</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 22, 'color': '#1f2937'}
    },
    xaxis=dict(
        title='Width (pixels)',
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1,
        zeroline=False,
        title_font=dict(size=14, color='#374151')
    ),
    yaxis=dict(
        title='Height (pixels)',
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1,
        zeroline=False,
        title_font=dict(size=14, color='#374151')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=700,
    showlegend=True,
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.95)',
        bordercolor='#d1d5db',
        borderwidth=1,
        font=dict(size=11)
    ),
    hovermode='closest'
)

fig.show()

# Enhanced summary statistics
image_size = (width_max, round(width_max/common_ratio))
print(f'\n{"="*50}')
print(f'📊 RECOMMENDED IMAGE SIZE: {image_size[0]}x{image_size[1]}')
print(f'{"="*50}')
print(f'   Aspect Ratio: {common_ratio:.2f}:1 (4:3)')
print(f'   Total Images: {len(metadata):,}')
print(f'   Width Range: {metadata["width"].min()}-{metadata["width"].max()}px')
print(f'   Height Range: {metadata["height"].min()}-{metadata["height"].max()}px')
print(f'{"="*50}\n')

In [ ]:
# Create 2D histogram/heatmap of image dimensions
fig = px.density_heatmap(
    metadata,
    x='width',
    y='height',
    nbinsx=50,
    nbinsy=50,
    title='<b>Image Dimension Density Heatmap</b>',
    labels={'width': 'Width (pixels)', 'height': 'Height (pixels)'},
    color_continuous_scale='Blues'
)

fig.update_layout(
    width=900,
    height=700,
    plot_bgcolor='white'
)

fig.show()

### Images exploration

#### All images

In [ ]:
# Print 5 example images of each class (+-3min)
for label in os.listdir(directory):
    path = directory + str(label)

    if not os.path.isdir(path):
        print(f"Directory {path} does not exist.")
        continue

    folder_data = os.listdir(path)
    k = 0
    print(f'{label} ({len(folder_data)} images)')

    # Collect image paths
    image_paths = []
    for image_path in folder_data:
        if k < 5:                                               # <-- change how many images per class
            full_path = os.path.join(path, image_path)
            image_paths.append(full_path)
            k += 1

    # Display images
    if image_paths:
        fig, axes = plt.subplots(1, len(image_paths), figsize=(15, 3))
        if len(image_paths) == 1:
            axes = [axes]

        for ax, img_path in zip(axes, image_paths):
            img = PilImage.open(img_path)
            ax.imshow(img)
            ax.axis('off')

        plt.tight_layout()
        plt.show()

From printing some examples of the images that we'll be working with we can see that we have a few:

- X-ray imagaes
- Images of signs
- Paintings
- Text extracts with no images
- Images unrelated to the class
- Maps
- Varying zoom / color / rotations

#### Grayscale VS CMYK

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

# Get only greyscale images
temp_generator = datagen.flow_from_dataframe(
    dataframe=metadata[metadata.color_channel == 'L'],
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # Not shuffling
)

In [ ]:
# Get class names from the generator
class_names = list(temp_generator.class_indices.keys())

# Determine the total number of batches in the generator
num_batches = int(math.ceil(temp_generator.n / temp_generator.batch_size))

for i in range(num_batches):
    images, labels = next(temp_generator)

    # Plot the images in the current batch
    batch_size_actual = images.shape[0]
    n_cols = min(8, batch_size_actual)                  # <-- set max columns per row
    n_rows = math.ceil(batch_size_actual / n_cols)

    plt.figure(figsize=(3 * n_cols, 3 * n_rows))

    for j in range(batch_size_actual):
        ax = plt.subplot(n_rows, n_cols, j + 1)
        plt.imshow(images[j])

        # Get the index of the highest probability to find the class name
        label_idx = np.argmax(labels[j])
        plt.title(class_names[label_idx], fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

Some images retain at least the shape of the actual animal, but most of them are really bad

In [ ]:
# Do the same for CMYK
temp_generator = datagen.flow_from_dataframe(
    dataframe=metadata[metadata.color_channel == 'CMYK'],
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # Not shuffling
)

In [ ]:
# Get class names from the generator
class_names = list(temp_generator.class_indices.keys())

# Determine the total number of batches in the generator
num_batches = int(math.ceil(temp_generator.n / temp_generator.batch_size))

for i in range(num_batches):
    images, labels = next(temp_generator)

    # Plot the images in the current batch
    batch_size_actual = images.shape[0]
    n_cols = min(8, batch_size_actual)                  # <-- set max columns per row
    n_rows = math.ceil(batch_size_actual / n_cols)

    plt.figure(figsize=(3 * n_cols, 3 * n_rows))

    for j in range(batch_size_actual):
        ax = plt.subplot(n_rows, n_cols, j + 1)
        plt.imshow(images[j])

        # Get the index of the highest probability to find the class name
        label_idx = np.argmax(labels[j])
        plt.title(class_names[label_idx], fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Changing the cmyk outliers to rgb in the metadata

metadata.loc[metadata['family'].isin(['chordata_emydidae', 'chordata_psittacidae']) & 
             (metadata['color_channel'] == 'CMYK'), 'color_channel'] = 'RGB'

Most of the images apper to be x-rays of sculls of different animals which don't really help identify the animals by the picture, and will more likely only confuse the model. \
(and there are 2 different images for some reason: 2 parrots and a turtle)

In [ ]:
# Delete the generator to free up resources (small impactbut why not)
del temp_generator

## Preprocessing

## Calculate image embeddings and split

In [ ]:
# Remove CMYK and grayscale images from metadata before split
print(f"Original metadata size: {len(metadata)}")
metadata = metadata[~metadata['color_channel'].isin(['CMYK', 'L'])].reset_index(drop=True)
print(f"After removing CMYK/grayscale: {len(metadata)}")

In [ ]:
# TRY TO ONLY USE RGB
# metadata = metadata[metadata.color_channel == 'RGB'].reset_index(drop=True)

In [ ]:
# good_images_df = good_images_df[good_images_df.family != 'formicidae'].reset_index(drop=True)

## Outlier detection / Removal

### Outlier detection using YOLO

In [ ]:
'''model = YOLO('yolov8n.pt')  # load the model'''

In [ ]:
'''from ultralytics import YOLO
import os
from PIL import Image as PilImage

MODEL_ID = 'yolov8s.pt'
CONFIDENCE_THRESHOLD = 0.85
PERSON_CLASS_ID = 0'''

In [ ]:
'''def find_images_with_people(metadata, directory='rare_species/', conf_threshold=CONFIDENCE_THRESHOLD):
    """Find images containing people using YOLO."""
    print(f"Loading {MODEL_ID}...")
    model = YOLO(MODEL_ID)
    
    has_person = []
    person_confidence = []
    
    print(f"Scanning for people with confidence > {conf_threshold}...")
    
    for idx, row in tqdm(metadata.iterrows(), total=len(metadata), desc="YOLO person detection"):
        filepath = os.path.join(directory, row['file_path'])
        
        try:
            results = model.predict(
                filepath,
                verbose=False,
                conf=conf_threshold,
                iou=0.5,
                classes=[PERSON_CLASS_ID]
            )
            
            boxes = results[0].boxes if results and results[0].boxes is not None else None
            
            if boxes and len(boxes) > 0:
                has_person.append(True)
                person_confidence.append(float(boxes.conf.max()))
            else:
                has_person.append(False)
                person_confidence.append(0.0)
                
        except Exception as e:
            has_person.append(False)
            person_confidence.append(0.0)
    
    return has_person, person_confidence'''

In [ ]:
'''# Run YOLO person detection
metadata['has_person'], metadata['person_confidence'] = find_images_with_people(metadata, directory=directory)

print(f"\nYOLO Results:")
print(f"  Images with people: {metadata['has_person'].sum()}")
print(f"  Images without people: {(~metadata['has_person']).sum()}")'''

### Outlier detection using CLIP

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

In [ ]:
# Check for GPU/MPS availability
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"\nUsing device: {device}")

# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Define semantic categories - "good" images first, then "bad" noise types
text_prompts = [
    # Good categories (what we WANT to keep) - indices 0-2
    "a photograph of an animal",
    "a wildlife photograph",
    "a photo of an animal in nature",
    
    # Bad categories (what we WANT to filter out) - indices 3+
    "an x-ray image",
    "a medical scan",
    "a drawing or sketch",
    "a map or diagram",
    "text document or book page",
    "a logo or icon",
    "a cooked meal",
    "prepared food",
    "a human",
    "a photo of a person",
    "a portrait of a human face",
    "people in a photograph",
    "a group of people",
    "a scientist or researcher",
]

# Precompute text embeddings
text_inputs = clip_processor(text=text_prompts, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

In [ ]:
def compute_clip_scores_batch(metadata, directory="rare_species/", batch_size=32):
    """Process all images and compute CLIP semantic scores."""
    all_results = []
    file_paths = metadata['file_path'].tolist()
    
    for i in tqdm(range(0, len(file_paths), batch_size), desc="Computing CLIP scores"):
        batch_paths = file_paths[i:i+batch_size]
        batch_images = []
        batch_indices = []
        
        for j, path in enumerate(batch_paths):
            try:
                full_path = directory + path
                img = PilImage.open(full_path).convert("RGB")
                batch_images.append(img)
                batch_indices.append(i + j)
            except Exception as e:
                all_results.append({
                    'clip_photo_score': None,
                    'clip_noise_score': None,
                    'clip_semantic_quality': None,
                    'clip_best_match': None
                })
        
        if not batch_images:
            continue
        
        image_inputs = clip_processor(images=batch_images, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad():
            image_features = clip_model.get_image_features(**image_inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            similarities = (image_features @ text_features.T).cpu().numpy()
        
        for sims in similarities:
            photo_score = sims[:3].max()
            noise_score = sims[3:].max()
            all_results.append({
                'clip_photo_score': float(photo_score),
                'clip_noise_score': float(noise_score),
                'clip_semantic_quality': float(photo_score - noise_score),
                'clip_best_match': text_prompts[sims.argmax()]
            })
    
    return pd.DataFrame(all_results)

In [ ]:
# Run CLIP scoring on your metadata
print("Computing CLIP semantic scores...")
clip_results = compute_clip_scores_batch(metadata, directory=directory, batch_size=32)

# Add CLIP columns to metadata
metadata['clip_photo_score'] = clip_results['clip_photo_score']
metadata['clip_noise_score'] = clip_results['clip_noise_score']
metadata['clip_semantic_quality'] = clip_results['clip_semantic_quality']
metadata['clip_best_match'] = clip_results['clip_best_match']

print(f"Processed {len(metadata)} images")

### Imagenet outlier detection

In [ ]:
'''from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions

print("\nLoading InceptionV3 model...")
imagenet_model = InceptionV3(weights='imagenet')
print("Model loaded successfully")'''

In [ ]:
# ImageNet animal detection helper function
'''def check_animal_presence(img_path, model, threshold=0.02):
    """
    Check if an image contains an animal using ImageNet predictions.
    
    Args:
        img_path: Path to the image
        model: InceptionV3 model
        threshold: Minimum probability to consider as animal
    
    Returns:
        is_animal: Boolean - True if animal detected
        animal_prob: Float - Highest animal class probability
        top_class: String - Top predicted class name
        top_prob: Float - Top prediction probability
    """
    from tensorflow.keras.preprocessing import image
    from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions
    
    # ImageNet animal class indices (classes 0-397 are animals)
    # Full list: https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a
    ANIMAL_CLASS_RANGE = (0, 398)  # Classes 0-397 are animals in ImageNet
    
    try:
        # Load and preprocess image
        img = image.load_img(img_path, target_size=(299, 299))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        
        # Get predictions
        preds = model.predict(x, verbose=0)
        decoded = decode_predictions(preds, top=10)[0]
        
        # Get top prediction
        top_class = decoded[0][1]
        top_prob = float(decoded[0][2])
        
        # Check if any animal class has high probability
        animal_prob = 0.0
        is_animal = False
        
        for pred in decoded:
            class_idx = pred[0]
            class_name = pred[1]
            prob = float(pred[2])
            
            # ImageNet classes 0-397 are animals
            # We can also check by keywords
            animal_keywords = [
                'dog', 'cat', 'bird', 'fish', 'snake', 'turtle', 'frog',
                'insect', 'butterfly', 'spider', 'crab', 'lobster', 'snail',
                'worm', 'jellyfish', 'coral', 'lion', 'tiger', 'bear',
                'elephant', 'monkey', 'ape', 'whale', 'dolphin', 'seal',
                'penguin', 'eagle', 'owl', 'parrot', 'lizard', 'crocodile',
                'salamander', 'scorpion', 'beetle', 'bee', 'ant', 'fly',
                'cockroach', 'mantis', 'dragonfly', 'moth', 'slug'
            ]
            
            # Check if it's likely an animal
            if any(keyword in class_name.lower() for keyword in animal_keywords):
                if prob > animal_prob:
                    animal_prob = prob
                    is_animal = True
        
        # If no keyword match, use class index range
        if not is_animal and top_prob > threshold:
            # Approximate check: most animals are in lower class indices
            # This is a heuristic - adjust as needed
            pass
        
        is_animal = animal_prob >= threshold
        
        return is_animal, animal_prob, top_class, top_prob
        
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return False, 0.0, "error", 0.0

print("✅ check_animal_presence function defined")'''

In [ ]:
'''# Run ImageNet animal detection
is_animal_list = []
animal_prob_list = []
top_class_list = []
top_prob_list = []

for idx, row in tqdm(metadata.iterrows(), total=len(metadata), desc="ImageNet animal detection"):
    img_path = os.path.join(directory, row['file_path'])
    is_animal, animal_prob, top_class, top_prob = check_animal_presence(
        img_path, imagenet_model, threshold=0.02
    )
    is_animal_list.append(is_animal)
    animal_prob_list.append(animal_prob)
    top_class_list.append(top_class)
    top_prob_list.append(top_prob)

metadata['has_animal'] = is_animal_list
metadata['animal_confidence'] = animal_prob_list
metadata['predicted_class'] = top_class_list
metadata['prediction_confidence'] = top_prob_list

print(f"\nImageNet Results:")
print(f"  Images with animals: {metadata['has_animal'].sum()}")
print(f"  Images without animals: {(~metadata['has_animal']).sum()}")'''

### Visualisations / Comparisons

In [ ]:
def show_all_clip_outliers(metadata, directory="rare_species/", n_cols=10):
    """Display ALL CLIP outliers."""
    bad_images = metadata[metadata['clip_semantic_quality'] < 0].copy()
    bad_images = bad_images.sort_values('clip_semantic_quality', ascending=True)
    
    if len(bad_images) == 0:
        print("No CLIP outliers found!")
        return
    
    print(f"Showing ALL {len(bad_images)} CLIP outliers\n")
    print("Category breakdown:")
    print(bad_images['clip_best_match'].value_counts())
    print("\n" + "="*60 + "\n")
    
    n_rows = math.ceil(len(bad_images) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2.5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, (_, row) in enumerate(bad_images.iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{row['clip_best_match'].replace('a ', '').replace('an ', '')[:15]}\n"
                f"{row['clip_semantic_quality']:.2f}",
                fontsize=6
            )
        except:
            pass
        axes[idx].axis('off')
    
    for idx in range(len(bad_images), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"CLIP Outliers ({len(bad_images)} images)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
def show_yolo_detections(metadata, directory="rare_species/", n_cols=10):
    """Display images where YOLO detected people."""
    person_images = metadata[metadata['has_person'] == True].copy()
    person_images = person_images.sort_values('person_confidence', ascending=False)
    
    if len(person_images) == 0:
        print("No images with people detected!")
        return
    
    print(f"Showing ALL {len(person_images)} images with people detected\n")
    
    n_rows = math.ceil(len(person_images) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2.5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, (_, row) in enumerate(person_images.iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(f"conf: {row['person_confidence']:.2f}", fontsize=6)
        except:
            pass
        axes[idx].axis('off')
    
    for idx in range(len(person_images), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"YOLO Person Detections ({len(person_images)} images)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
def show_non_animal_images(metadata, directory="rare_species/", n_cols=10, n_rows=5):
    """Display images where ImageNet didn't detect animals."""
    non_animals = metadata[metadata['has_animal'] == False].copy()
    non_animals = non_animals.sort_values('animal_confidence', ascending=True)
    
    if len(non_animals) == 0:
        print("All images contain animals!")
        return
    
    n_show = min(n_cols * n_rows, len(non_animals))
    print(f"Showing {n_show} of {len(non_animals)} non-animal images\n")
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2.5*n_rows))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(non_animals.head(n_show).iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{row['predicted_class'][:12]}\n{row['animal_confidence']:.3f}",
                fontsize=6
            )
        except:
            pass
        axes[idx].axis('off')
    
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"ImageNet Non-Animal Images ({len(non_animals)} total)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Show all outliers
'''print("\n" + "="*60)
print("VISUALIZING OUTLIERS")
print("="*60)

show_all_clip_outliers(metadata, directory=directory)
show_yolo_detections(metadata, directory=directory)
show_non_animal_images(metadata, directory=directory)'''

### Outlier removal

In [ ]:
print(f"Before filtering: {len(metadata)} images")

outliers = metadata[metadata['clip_semantic_quality'] < 0].copy()
metadata = metadata[metadata['clip_semantic_quality'] >= 0].reset_index(drop=True)

print(f"Removed: {len(outliers)} outliers")
print(f"After filtering: {len(metadata)} images")

### Import / split the dataset

In [ ]:
print(f"\nNumber of families: {metadata['family'].nunique()}")
low_images_class = metadata['family'].value_counts().idxmin()
print(f"Class with fewest images: '{low_images_class}' with {metadata['family'].value_counts().min()} images")

# First split: train+val / test
train_df, test_df = train_test_split(
    metadata,
    test_size=0.1,
    stratify=metadata['family'],
    shuffle=True,
    random_state=42
)

# Second split: train / val
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    stratify=train_df['family'],
    shuffle=True,
    random_state=42
)

# Print the final proportions of the split
total = len(train_df) + len(val_df) + len(test_df)
print(f"\nSplit complete:")
print(f"  Training  : {len(train_df)} ({len(train_df)/total*100:.1f}%)")
print(f"  Validation: {len(val_df)} ({len(val_df)/total*100:.1f}%)")
print(f"  Testing   : {len(test_df)} ({len(test_df)/total*100:.1f}%)")
print(f"  Total     : {total}")

In [ ]:
# Set image size
image_size = (500, 375)

from tensorflow.keras.applications.efficientnet import preprocess_input

# MINIMAL augmentation - let the model learn first
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    fill_mode='nearest'
)

# Validation/Test generator (no augmentation)
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Create basic generator to get class_indices
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    shuffle=False
)

N_CLASSES = len(train_generator.class_indices)
print(f"Found {N_CLASSES} classes")

In [ ]:
#  Critical debug check
images, labels = next(train_generator)

print(f"\n🔍 VALIDATION CHECKS:")
print(f"Image batch shape: {images.shape}")
print(f"Image value range: [{images.min():.3f}, {images.max():.3f}]")
print(f"Expected range for EfficientNet: roughly [-2.1, 2.6]")
print(f"Labels shape: {labels.shape}")
print(f"Labels range: [{labels.min()}, {labels.max()}]")
print(f"Unique labels in batch: {len(np.unique(labels))}")
print(f"Any all-zero images: {(images.sum(axis=(1,2,3)) == 0).any()}")

# Check if values look reasonable
if images.min() < -3 or images.max() > 3:
    print("⚠️ WARNING: Image values outside expected range!")

In [ ]:
# Get a fresh batch
images, labels = next(train_generator)

# Get class names
class_names = list(train_generator.class_indices.keys())

# Plot the images
plt.figure(figsize=(12, 8))

for i in range(min(21, len(images))):
    ax = plt.subplot(3, 7, i + 1)
    # Denormalize for visualization
    img_display = images[i].copy()
    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())
    plt.imshow(img_display)
    plt.title(class_names[int(labels[i])], fontsize=8)
    plt.axis("off")

plt.suptitle("Sample Training Batch (minimal augmentation)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Model


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

# Keras imports
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization,
    Input, Concatenate, LeakyReLU, Embedding, Flatten
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import keras_tuner as kt

# Check GPU
print("TensorFlow version:", tf.__version__)
print("GPUs Available:", tf.config.list_physical_devices('GPU'))

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
def plot_model_results(history, metric='loss'):
    """Plot training and validation metrics."""
    plt.figure(figsize=(10, 5))
    
    plt.plot(history.history[metric], label=f'Training {metric}', linewidth=2)
    plt.plot(history.history[f'val_{metric}'], label=f'Validation {metric}', linewidth=2)
    
    plt.title(f'Model {metric.capitalize()}', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel(metric.capitalize())
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    if metric == 'loss':
        best_epoch = np.argmin(history.history['val_loss'])
        best_val = history.history['val_loss'][best_epoch]
        print(f"Best val_loss: {best_val:.4f} at epoch {best_epoch + 1}")
    else:
        best_epoch = np.argmax(history.history['val_accuracy'])
        best_val = history.history['val_accuracy'][best_epoch]
        print(f"Best val_accuracy: {best_val:.4f} ({best_val*100:.2f}%) at epoch {best_epoch + 1}")

In [ ]:
N_CLASSES = len(train_generator.class_indices)
print(f"Number of classes: {N_CLASSES}")

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weight range: {min(class_weights):.3f} - {max(class_weights):.3f}")

In [ ]:
class MultiInputGenerator(tf.keras.utils.Sequence):
    """Generator that yields {'image_input': images, 'phylum_input': phylum} and labels."""
    
    def __init__(self, dataframe, directory, phylum_array, image_datagen, 
                 batch_size=32, target_size=(224, 224), class_indices=None, shuffle=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.directory = directory
        self.phylum_array = phylum_array
        self.image_datagen = image_datagen
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.indices = np.arange(len(dataframe))
        
        if class_indices is not None:
            self.class_indices = class_indices
        else:
            classes = sorted(dataframe['family'].unique())
            self.class_indices = {cls: i for i, cls in enumerate(classes)}
        
        self.on_epoch_end()
        
    def __len__(self):
        return int(np.ceil(len(self.dataframe) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        
        images = []
        labels = []
        phylum_batch = []
        
        for i in batch_indices:
            img_path = self.directory + '/' + self.dataframe.iloc[i]['file_path']
            img = tf.keras.utils.load_img(img_path, target_size=self.target_size)
            img_array = tf.keras.utils.img_to_array(img)
            
            img_array = self.image_datagen.random_transform(img_array)
            img_array = self.image_datagen.standardize(img_array)
            
            images.append(img_array)
            
            family = self.dataframe.iloc[i]['family']
            labels.append(self.class_indices[family])
            phylum_batch.append(self.phylum_array[i])
        
        images = np.array(images)
        labels = np.array(labels)
        phylum_batch = np.array(phylum_batch).reshape(-1, 1)
        
        return {'image_input': images, 'phylum_input': phylum_batch}, labels
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [ ]:
# ============================================================
# ENCODE PHYLUM FOR EMBEDDING (optional auxiliary input)
# ============================================================

phylum_encoder = LabelEncoder()
metadata['phylum_encoded'] = phylum_encoder.fit_transform(metadata['phylum'])

n_phylums = metadata['phylum'].nunique()
print(f"\nNumber of unique phylums: {n_phylums}")
print(f"Phylums: {list(phylum_encoder.classes_)}")

# Apply encoding to splits
train_df['phylum_encoded'] = phylum_encoder.transform(train_df['phylum'])
val_df['phylum_encoded'] = phylum_encoder.transform(val_df['phylum'])
test_df['phylum_encoded'] = phylum_encoder.transform(test_df['phylum'])

# Create arrays for the metadata input
train_phylum = train_df['phylum_encoded'].values
val_phylum = val_df['phylum_encoded'].values
test_phylum = test_df['phylum_encoded'].values

# Create multi-input generators
train_multi_gen = MultiInputGenerator(
    dataframe=train_df,
    directory=directory,
    phylum_array=train_phylum,
    image_datagen=train_datagen,
    batch_size=32,
    target_size=image_size,
    class_indices=train_generator.class_indices,
    shuffle=True
)

val_multi_gen = MultiInputGenerator(
    dataframe=val_df,
    directory=directory,
    phylum_array=val_phylum,
    image_datagen=val_test_datagen,
    batch_size=32,
    target_size=image_size,
    class_indices=train_generator.class_indices,
    shuffle=False
)

test_multi_gen = MultiInputGenerator(
    dataframe=test_df,
    directory=directory,
    phylum_array=test_phylum,
    image_datagen=val_test_datagen,
    batch_size=32,
    target_size=image_size,
    class_indices=train_generator.class_indices,
    shuffle=False
)

print(f"✅ Multi-input generators created")

In [ ]:
def build_improved_model(hp):
    """
    Image model with phylum embedding as auxiliary input.
    The embedding is concatenated with image features before dense layers.
    """
    # Base model
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False
    
    # IMAGE INPUT
    image_input = Input(shape=(224, 224, 3), name='image_input')
    
    x = base_model(image_input, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    
    # PHYLUM INPUT (auxiliary)
    phylum_input = Input(shape=(1,), name='phylum_input')
    embed_dim = hp.Int('embed_dim', min_value=8, max_value=32, step=8)
    phylum_embed = Embedding(input_dim=n_phylums, output_dim=embed_dim)(phylum_input)
    phylum_embed = Flatten()(phylum_embed)
    
    # CONCATENATE image features + phylum embedding
    x = Concatenate()([x, phylum_embed])
    
    # Dense layers
    n_layers = hp.Int('n_layers', min_value=1, max_value=2, step=1)
    
    for i in range(n_layers):
        units = hp.Int(f'units_{i}', min_value=256, max_value=512, step=128)
        l2_reg = hp.Float(f'l2_{i}', min_value=0.01, max_value=0.1, step=0.01)
        
        x = Dense(units, kernel_regularizer=L2(l2_reg))(x)
        x = BatchNormalization()(x)
        x = LeakyReLU(negative_slope=0.1)(x)
        
        dropout = hp.Float(f'dropout_{i}', min_value=0.4, max_value=0.6, step=0.1)
        x = Dropout(dropout)(x)
    
    outputs = Dense(N_CLASSES, activation='softmax')(x)
    
    model = Model(inputs=[image_input, phylum_input], outputs=outputs)
    
    lr = hp.Choice('learning_rate', values=[5e-4, 1e-4, 5e-5])
    
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [ ]:
# ============================================================
# HYPERBAND SEARCH
# ============================================================

print("="*60)
print("HYPERBAND SEARCH")
print("="*60)

tuner = kt.Hyperband(
    build_improved_model,
    objective='val_accuracy',
    max_epochs=15,
    factor=3,
    directory='keras_tuner',
    project_name='improved_model_v1',
    overwrite=True
)

tuner.search(
    train_multi_gen,
    epochs=15,
    class_weight=class_weight_dict,
    validation_data=val_multi_gen,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=1
)

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\nBest Hyperparameters:")
print(best_hp.values)
BEST_HP = best_hp.values


In [ ]:
# ============================================================
# BUILD FINAL MODEL
# ============================================================

print("="*60)
print("BUILDING FINAL MODEL")
print("="*60)

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

# IMAGE INPUT
image_input = Input(shape=(224, 224, 3), name='image_input')

x = base_model(image_input, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)

# PHYLUM INPUT
phylum_input = Input(shape=(1,), name='phylum_input')
embed_dim = BEST_HP.get('embed_dim', 16)
phylum_embed = Embedding(input_dim=n_phylums, output_dim=embed_dim)(phylum_input)
phylum_embed = Flatten()(phylum_embed)

print(f"\nPhylum embedding: {n_phylums} phylums -> {embed_dim} dimensions")

# CONCATENATE
x = Concatenate()([x, phylum_embed])

n_layers = BEST_HP.get('n_layers', 1)
print(f"Building with {n_layers} dense layers:")

for i in range(n_layers):
    units = BEST_HP.get(f'units_{i}', 256)
    l2_reg = BEST_HP.get(f'l2_{i}', 0.05)
    dropout = BEST_HP.get(f'dropout_{i}', 0.4)
    
    x = Dense(units, kernel_regularizer=L2(l2_reg))(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    x = Dropout(dropout)(x)
    
    print(f"  Layer {i+1}: Dense({units}) + BN + LeakyReLU + Dropout({dropout}), L2={l2_reg}")

outputs = Dense(N_CLASSES, activation='softmax')(x)
model = Model(inputs=[image_input, phylum_input], outputs=outputs)

print(f"\nTotal params: {model.count_params():,}")

In [ ]:
# ============================================================
# PHASE 1: FROZEN BASE
# ============================================================

print("="*60)
print("PHASE 1: TRAINING WITH FROZEN BASE")
print("="*60)

model.compile(
    optimizer=Adam(learning_rate=BEST_HP.get('learning_rate', 1e-4)),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p1 = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model_phase1.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

history_p1 = model.fit(
    train_multi_gen,
    epochs=40,
    class_weight=class_weight_dict,
    validation_data=val_multi_gen,
    callbacks=callbacks_p1,
    verbose=1
)

In [ ]:
# ============================================================
# PHASE 2: FINE-TUNING (MORE CONSERVATIVE)
# ============================================================

print("="*60)
print("PHASE 2: FINE-TUNING")
print("="*60)

# KEY CHANGE: Unfreeze fewer layers (20 instead of 30)
base_model.trainable = True
UNFREEZE_LAYERS = 20

for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

print(f"Unfroze last {UNFREEZE_LAYERS} layers")

# KEY CHANGE: Even lower learning rate for fine-tuning
model.compile(
    optimizer=Adam(learning_rate=5e-6),  # Lower than before
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-8, verbose=1),
    ModelCheckpoint('best_model_phase2.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

history_p2 = model.fit(
    train_multi_gen,
    epochs=15,
    class_weight=class_weight_dict,
    validation_data=val_multi_gen,
    callbacks=callbacks_p2,
    verbose=1
)

In [ ]:
# ============================================================
# PLOT TRAINING HISTORY
# ============================================================

def plot_history(history, title_prefix=""):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    axes[0].plot(history.history['loss'], label='Train', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val', linewidth=2)
    axes[0].set_title(f'{title_prefix}Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history.history['accuracy'], label='Train', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Val', linewidth=2)
    axes[1].set_title(f'{title_prefix}Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print gap
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    gap = final_train_acc - final_val_acc
    print(f"Final train/val gap: {gap:.4f} ({gap*100:.2f}%)")

print("\n" + "="*60)
print("TRAINING HISTORY")
print("="*60)
plot_history(history_p1, "Phase 1: ")
plot_history(history_p2, "Phase 2: ")

In [ ]:
# ============================================================
# EVALUATION
# ============================================================

print("="*60)
print("FINAL EVALUATION")
print("="*60)

test_generator.reset()
test_preds = model.predict(test_multi_gen, verbose=1)
y_pred = np.argmax(test_preds, axis=1)
y_true = test_generator.classes

test_acc = np.mean(y_pred == y_true)
test_f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\n🎯 Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"🎯 F1 Score: {test_f1:.4f}")

In [ ]:
# ============================================================
# TOP-K ACCURACY
# ============================================================

print("\n" + "="*60)
print("TOP-K ACCURACY")
print("="*60)

def top_k_accuracy(y_true, y_pred_proba, k):
    top_k = np.argsort(y_pred_proba, axis=1)[:, -k:]
    correct = [y_true[i] in top_k[i] for i in range(len(y_true))]
    return np.mean(correct)

for k in [1, 3, 5, 10]:
    acc = top_k_accuracy(y_true, test_preds, k)
    print(f"Top-{k} Accuracy: {acc:.4f} ({acc*100:.2f}%)")

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print(f"\nCorrectly classified: {np.trace(cm)} / {cm.sum()} ({np.trace(cm)/cm.sum()*100:.2f}%)")


In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================

print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

import pickle

model.save('image_model_final.keras')
print("✓ Saved: image_model_final.keras")

# Save hyperparameters
with open('best_hyperparameters.pkl', 'wb') as f:
    pickle.dump(BEST_HP, f)
print("✓ Saved: best_hyperparameters.pkl")

# Save class indices for inference
with open('class_indices.pkl', 'wb') as f:
    pickle.dump(train_generator.class_indices, f)
print("✓ Saved: class_indices.pkl")

# Save phylum encoder for inference
with open('phylum_encoder.pkl', 'wb') as f:
    pickle.dump(phylum_encoder, f)
print("✓ Saved: phylum_encoder.pkl")

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("🎯 FINAL SUMMARY")
print("="*60)

print(f"""
Architecture:
  - Pretrained: EfficientNetB0 (ImageNet weights)
  - Phylum embedding: {n_phylums} phylums -> {BEST_HP.get('embed_dim', 16)} dims
  - Custom dense layers: {BEST_HP.get('n_layers', 1)} layer(s) with LeakyReLU
  - Fine-tuned: Last {UNFREEZE_LAYERS} layers
  
Regularization:
  - Dropout: {BEST_HP.get('dropout_0', 0.4)}
  - L2: {BEST_HP.get('l2_0', 0.05)}
  - Label smoothing: 0.1
  
Training:
  - Phase 1: Frozen base, LR={BEST_HP.get('learning_rate', 1e-4)}
  - Phase 2: Fine-tuned with LR=5e-6

Results:
  - Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)
  - F1 Score: {test_f1:.4f}
  - Top-5 Accuracy: {top_k_accuracy(y_true, test_preds, 5):.4f}
""")